# Structured output from text

## Structured output från fritext – nästlade modeller

Målet är att plocka ut strukturerad information ur en ostrukturerad text (`raw_text.txt`).

- **`Person`** beskriver en medverkande: namn, organisation och ämne.
- **`Resilience`** beskriver hela evenemanget. Fältet `contributors: list[Person]` är en **nästlad modell**, alltså en lista där varje element själv är en Pydantic-modell. LLM:en måste då leverera en lista med objekt som alla klarar valideringen.
- **`instructions`** sätter rollen. `output_type=Resilience` tvingar fram strukturen.

**Obs:** `Field("text")` sätter texten som **standardvärde**, inte som beskrivning. LLM:en får alltså ingen instruktion för fältet, och om den hoppar över fältet blir värdet bara texten. Rätt skrivsätt är `Field(description="text")`.

In [1]:
from model import nvidia_model
from pydantic_ai import Agent
from pydantic import BaseModel, Field

class Person(BaseModel):
    name: str
    org: str = Field("Company or organisation")
    subject: str = Field(description="the subject the person has")

class Resilience(BaseModel):
    title: str = Field(description="pick suitable headline from the input, don't hallucinate")
    description: str = Field("3-5 sentences summary of the content")
    contributors: list[Person]
    contact: str = Field(description="contact person for this event")
    org_nr: str = Field(description="pick out the organization number in form of xxxxxx-xxxx")

with open("raw_text.txt", encoding="utf-8") as file:
    raw_text = file.read()

print(raw_text)

agent = Agent(model=nvidia_model("nvidia/nemotron-3.5-super-vl-120b-a12b"),
            instructions="You are an expert in summarizing content in Swedish. "
                        "Take the provided content and summarize it according to output type"
                        )
result = await agent.run(raw_text, output_type=Resilience)


STI
 
Skip to content
Program
Kurser
Så söker du
För företag
Läsvärt
Om STI
Kontakt
STI RESILIENT STOCKHOLM · temaVECKA 48 2026
Vilken kompetens behövs när förutsättningarna förändras?
Under en temavecka i november samlas studerande från STI:s utbildningar kring en gemensam fråga: Hur gör vi Stockholm mer resilient inför framtidens utmaningar?

 
STI Resilient Stockholm 2026

En temavecka om framtidens utmaningar
Världen förändras snabbt, och framtidens arbetsliv ställer nya krav på både kompetens och samarbete. Att bygga ett samhälle som kan möta förändring kräver att vi kan tänka nytt, arbeta över gränser och se utmaningar ur flera perspektiv.

Under Resilient Stockholm den 23–27 november får studerande från STI:s olika utbildningar utforska hur deras framtida yrkesroller kan bidra till ett mer resilient Stockholm.

Målet är att ge dem nya perspektiv och verktyg som de kan ta med sig in i sina framtida yrkesroller.


Vad menar vi med resiliens?
Resiliens handlar om förmågan att möta 

In [2]:
print(result.output)

title='STI Resilient Stockholm 2026' description='Under tema veckan STI Resilient Stockholm vecka 48 2026 möts studerande från STI:s olika utbildningar för att tillsammans ta fram lösningar på framtidens utmaningar för Stockholm, exempelvis skyfall, köldknäppar och värmeböljor. Under veckan arbetar de i tvärvetenskapliga lag och får input från arbetslivet och forskning genom externa föreläsare. Veckan avslutas med att lagen presenterar sina förslag för representanter från näringslivet och en vinnare utses. Syftet är att ge deltagarna nya perspektiv och verktyg för sina framtida yrkesroller.' contributors=[Person(name='Clara Ternström', org='Stockholm Exergi', subject='Hur Stockholm Exergi arbetar med resiliens'), Person(name='Robert Nilsson', org='Stockholm Exergi', subject='Hur Stockholm Exergi arbetar med resiliens'), Person(name='Joakim Netz', org='Jönköping University', subject='Om resiliens, innovation och förnyelse i tider av samhällskriser'), Person(name='Anna Körlof', org='Swec

In [3]:
content = result.output

In [4]:
content.org_nr

'Ej angivet'

In [5]:
content.contact

'AnnaStina Johansson'

## Structure a program

## Structure a program – samma mönster på ny data

Samma teknik används på en kursbeskrivning (`ux_kurser.txt`):
1. Definiera schemat (`Program`) med de fält du vill ha ut.
2. Läs in texten från fil.
3. Kör agenten med `output_type=Program`.

Resultatet blir ett `Program`-objekt i stället för fri text, så fälten kan användas direkt i kod, t.ex. `result.output.points`.

**Förbättringsidéer:**
- `course` borde vara `list[str]`, eftersom ett program har flera kurser.
- Samma `Field("...")`-problem som ovan: använd `description=`.

In [6]:
class Program(BaseModel):
    year: str = Field("Vilket år")
    name: str = Field("Programmets namn")
    course: str = Field("Programmets kurser")
    points: int
    description: str = Field("Vad kursen innehåller")


with open("ux_kurser.txt", encoding="utf-8") as file:
    ux_kurser = file.read()

agent = Agent(model=nvidia_model("nvidia/nemotron-3.5-super-vl-120b-a12b"),
            instructions="You are an expert in summarizing content in Swedish. "
                        "Take the provided content and summarize it according to output type",
                        )
result = await agent.run(ux_kurser, output_type=Program)

In [7]:
print(result.output)

year='År 1' name='YH-utbildning inom UX-design' course='Introduktion till studier och arbetsliv, Interaktionsdesign, Frontend, UX research, tjänstedesign och användbarhetstestning, Visuell design, Agil metodik och arbetssätt' points=400 description='Utbildningen omfattar 400 YH-poäng och kombinerar UX-metodik, research, interaktionsdesign, visuell design, teknik, kommunikation och praktik. År 1 (200 poäng) täcker grunderna i UX, metoder och praktiska verktyg: introduktion till studier och yrkesroll (10p), interaktionsdesign med kognition, GDPR, etik, AI och frontend (50p), frontend i HTML5, CSS3 och JavaScript (20p), UX research, tjänstedesign och användbarhetstestning med intervjuer, personas och prototyper (65p), visuell design med typografi, färglära, designsystem och tillgänglighet (30p), samt agil metodik med user stories, standups och UX-designerns roll i agila team (25p).'
